# ស៊ុមកម្មវិធីភ្នាក់ងារមៃក្រូសូហ្វ (Microsoft Agent Framework) — Azure OpenAI (Responses API)

នៅក្នុងគំរូកូដនេះ អ្នកនឹងប្រើ **ស៊ុមកម្មវិធីភ្នាក់ងារមៃក្រូសូហ្វ (MAF)** ដើម្បីបង្កើតភ្នាក់ងារងាយៗ មួយ ដែលគាំទ្រដោយ **Azure OpenAI** ដោយប្រើ **Responses API**។

> **កំណត់ចំណាំអំពីការផ្លាស់ប្តូរ៖** គំរូនេះឆ кардаប្រើ Semantic Kernel មាន GitHub Models មុននេះ។ វាត្រូវបានផ្លាស់ប្តូរ​ទៅស៊ុមកម្មវិធីភ្នាក់ងារមៃក្រូសូហ្វ ហើយ GitHub Models (ដែលបានរំលង ហើយនឹងឈប់ប្រើចាប់ពីខែកក្កដា ឆ្នាំ 2026) ត្រូវបានជំនួសដោយ Azure OpenAI ដែលគាំទ្រ Responses API។ `OpenAIChatClient` ក្នុង MAF ព្រមទាំង​គោលដៅទៅកាន់ផ្នែកចុងក្រោយ `/openai/v1/` ដែលមានស្ថេរភាពរបស់ Azure OpenAI ហើយប្រើ Responses API ជាប្រភេទលំនាំដើម។

គោលបំណងនៃគំរូនេះគឺដើម្បីបង្ហាញជំហាន ដែលនៅពេលក្រោយនឹងត្រូវអនុវត្តក្នុងគំរូកូដបន្ថែម ដែលអនុវត្តលំនាំ agentic ផ្សេងៗ។


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## នាំចូលកញ្ចប់ Python ដែលត្រូវការ


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## ការកំណត់ឧបករណ៍មួយ

ក្នុង Microsoft Agent Framework, **ឧបករណ៍** គឺជាអនុគមន៍ Python សាមញ្ញមួយដែលត្រូវបានពាក់ព័ន្ធជាមួយ `@tool` ដែលភ្នាក់ងារអាចហៅបាន។ ខាងក្រោមយើងកំណត់ឧបករណ៍មួយដែលត្រឡប់មកជាទីកន្លែងធ្វើដំណើរពេលឈប់សម្រាកមួយចៃដន្យ ហើយជៀសវាងក្រឡេចេញលើទីកន្លែងមុន។ 


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## ការបង្កើតភ្នាក់ងារ

ទីនេះ យើងបង្កើតភ្នាក់ងារដែលមានឈ្មោះ `TravelAgent`។

ក្នុងឧទាហរណ៍នេះ យើងប្រើការណែនាំមូលដ្ឋានខ្លះៗ។ អ្នកអាចផ្លាស់ប្តូរការណែនាំទាំងនេះ ដើម្បីសង្កេតឃើញពីរបៀបដំណើរការរបស់ភ្នាក់ងារប្រែប្រួលយ៉ាងដូចម្តេច។


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## ការបើកដំណើរការអេជន

ឥឡូវនេះយើងអាចបើកដំណើរការអេជនបាន។ យើងបង្កើត `AgentSession` ដើម្បីឲ្យអេជនចាំការសន្ទនារវាងជុំវាច្រើន ហើយបន្ទាប់មកផ្ញើ `user_inputs` ពីរម្លប់។ លេខមួយសួរពីការធ្វើដំណើរម្តង; លេខពីរបាននិយាយថាអ្នកប្រើមិនចូលចិត្តអនុសាសន៍នោះ ហើយសួរអំពីមិត្តភាពមួយទៀត — អេជនប្រើប្រវត្តិសម័យនៃសម័យចុះបញ្ជីរួមជាមួយឧបករណ៍ `get_random_destination` ដើម្បីឆ្លើយតប។

អ្នកអាចកែប្រែសារទាំងនេះដើម្បីសង្កេតមើលរបៀបទ្រង់ចិត្តរបស់អេជន។ ការឆ្លើយតបត្រូវបាន **ចាក់ផ្សាយ តាមToken** មួយដោយមួយ។


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
